# 07 Multi-Food Detection Exploration

This notebook explores a detector-first path for FoodLens. The current FoodLens model is a **single-label Food-101 classifier**; this notebook tests whether a pretrained detector can propose food regions that can later be classified crop by crop.

The notebook is intentionally separated from the classifier pipeline so detection quality can be reviewed before app integration.

## 1. Imports And Setup

All imports are centralized in this cell. The notebook keeps dependencies notebook-local so it can run on Kaggle after attaching the required inputs.

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import random

import cv2
import numpy as np
import pandas as pd
from PIL import Image

try:
    from ultralytics import YOLO
except ImportError as exc:
    raise ImportError(
        "Install ultralytics in the notebook environment before running "
        "detection."
    ) from exc

ImportError: Install ultralytics in the notebook environment before running detection.

## 2. Configuration

Set the input mode, attached Kaggle path, detector weights, thresholds, and output locations here. Use `MODE = "image"` for one still image or `MODE = "video"` for sampled video frames.

In [ ]:
@dataclass(frozen=True)
class Config:
    MODE: str = "image"  # image or video
    SEED: int = 42
    INPUT_PATH: Path = Path("/kaggle/input/foodlens-demo/sample.jpg")
    OUTPUT_DIR: Path = Path("/kaggle/working/results/multi_food_detection")
    DETECTOR_WEIGHTS: str = "yolo11n.pt"
    CONFIDENCE_THRESHOLD: float = 0.25
    IOU_THRESHOLD: float = 0.50
    MAX_DETECTIONS: int = 20
    VIDEO_FRAME_STRIDE_SECONDS: float = 2.0


CFG = Config()
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "crops").mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
CFG

## 3. Load Detector

A pretrained YOLO model is used first. This is an exploration step, not the final food-specific detector.

In [ ]:
detector = YOLO(CFG.DETECTOR_WEIGHTS)
detector

## 4. Load Image Or Sampled Video Frames

Video mode samples frames at a fixed interval so detection can be tested without processing every frame.

In [ ]:
def sample_video_frames(
    video_path: Path,
    stride_seconds: float,
) -> list[tuple[int, Image.Image]]:
    """Sample RGB frames from a video file.

    Args:
        video_path: Path to the input video.
        stride_seconds: Seconds between sampled frames.

    Returns:
        A list of `(frame_index, image)` tuples.
    """
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    frame_stride = max(1, int(fps * stride_seconds))
    frames: list[tuple[int, Image.Image]] = []
    frame_index = 0

    while True:
        success, frame_bgr = capture.read()
        if not success:
            break
        if frame_index % frame_stride == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append((frame_index, Image.fromarray(frame_rgb)))
        frame_index += 1

    capture.release()
    return frames


def load_inputs() -> list[tuple[str, Image.Image]]:
    """Load one image or sampled video frames for detection."""
    if CFG.MODE == "video":
        sampled_frames = sample_video_frames(
            CFG.INPUT_PATH,
            CFG.VIDEO_FRAME_STRIDE_SECONDS,
        )
        return [(f"frame_{idx:06d}", image) for idx, image in sampled_frames]
    return [(CFG.INPUT_PATH.stem, Image.open(CFG.INPUT_PATH).convert("RGB"))]


inputs = load_inputs()
len(inputs)

## 5. Run Detection And Export Crops

The detector exports both annotated figures and individual crops. These crops become the input to Notebook 8.

In [ ]:
def detect_and_export(
    source_name: str,
    image: Image.Image,
) -> list[dict[str, object]]:
    """Detect regions and export crop artifacts.

    Args:
        source_name: Stable image or frame identifier.
        image: RGB image to process.

    Returns:
        Detection metadata rows for CSV export.
    """
    result = detector.predict(
        source=np.array(image),
        conf=CFG.CONFIDENCE_THRESHOLD,
        iou=CFG.IOU_THRESHOLD,
        max_det=CFG.MAX_DETECTIONS,
        verbose=False,
    )[0]

    rows: list[dict[str, object]] = []
    boxes = result.boxes
    if boxes is None:
        return rows

    for detection_index, box in enumerate(boxes):
        x1, y1, x2, y2 = [int(value) for value in box.xyxy[0].tolist()]
        confidence = float(box.conf[0])
        detector_class_id = int(box.cls[0])
        detector_label = result.names.get(detector_class_id, str(detector_class_id))
        crop = image.crop((x1, y1, x2, y2))
        crop_path = (
            CFG.OUTPUT_DIR
            / "crops"
            / f"{source_name}_crop_{detection_index:02d}.jpg"
        )
        crop.save(crop_path, quality=95)

        rows.append(
            {
                "source_id": source_name,
                "detection_index": detection_index,
                "detector_label": detector_label,
                "detector_confidence": confidence,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "crop_path": str(crop_path),
            }
        )

    annotated = Image.fromarray(result.plot()[..., ::-1])
    figure_path = CFG.OUTPUT_DIR / "figures" / f"{source_name}_detections.jpg"
    annotated.save(figure_path, quality=95)
    return rows


detection_rows: list[dict[str, object]] = []
for source_name, source_image in inputs:
    detection_rows.extend(detect_and_export(source_name, source_image))

detections_df = pd.DataFrame(detection_rows)
detections_df.to_csv(CFG.OUTPUT_DIR / "detections.csv", index=False)
detections_df.head()

## 6. Review Outputs

Review `detections.csv`, exported crops, and annotated images before moving to classifier-per-crop integration. The key question is whether detector regions are clean enough for Food-101 crop classification.